# Семинар 1. Линейная регрессия

#### Шаг 1. Загрузка датасета и вывод на экран

In [ ]:
import pandas as pd

# Файл delivery_dataset.csv должен лежать рядом с ноутбуком
df = pd.read_csv("delivery_dataset.csv")

print(f"Размер датасета: {df.shape[0]} строк, {df.shape[1]} столбцов")
display(df.head())

#### Шаг2. Разделение выборки на тренировочную, валидационную и тестовую с выбором нужных столбцов:
* в тренировочной: 300 первых заказов;
* в валидационной: 100 следующих; 
* в тестовой: 100 последних.

In [ ]:
# ID и дата не используются как признаки: берем числовые характеристики заказа
features = [
    "Расстояние до клиента (в км)",
    "Количество позиций в чеке (в шт.)",
    "Балл пробок на дорогах (в баллах)",
    "Погода (в градусах Цельсия)",
    "Этаж доставки (в этажах)",
]
target = "Время доставки (в минутах)"

# Первые 300 — train, следующие 100 — val, последние 100 — test
train = df.iloc[:300]
val = df.iloc[300:400]
test = df.iloc[400:500]

X_train, y_train = train[features], train[target]
X_val, y_val = val[features], val[target]
X_test, y_test = test[features], test[target]

print("train:", X_train.shape, y_train.shape)
print("val:  ", X_val.shape, y_val.shape)
print("test: ", X_test.shape, y_test.shape)

#### Вопрос 1. 

Для чего мы разбиваем данные на выборки? Для чего нам нужна тренировачная, для чего валидационная, для чего тестовая?

**Ответ:**  
Тренировочная выборка нужна, чтобы модель подобрала веса. Валидационная — чтобы проверить её во время настройки на данных на которых она не обучалась. Тестовая — для финальной проверки уже готовой модели. Так мы не оцениваем модель на тех же данных, по которым она училась.

#### Шаг 3. Установка sklearn чтобы обучать модель линейной регрессии через Ridge

In [ ]:
%pip install -q scikit-learn

#### Шаг 4. Обучение модели линейной регрессии на тренировочной выборке с помощью Ridge и sparse_cg

In [ ]:
from sklearn.linear_model import Ridge

# alpha=0.0 — без регуляризации, solver="sparse_cg" — метод из лекции
model = Ridge(
    alpha=0.0,
    solver="sparse_cg",
    tol=1e-12,
    max_iter=100
)
model.fit(X_train, y_train)

print("Свободный коэффициент:", round(model.intercept_, 4))
print("Веса признаков:")
for name, weight in zip(features, model.coef_):
    print(f"  {name}: {weight:.4f}")

#### Шаг 5. Вывод на экран графиков ошибок (SSE, MSE, RMSE) в зависимости от итераций для train и val выборок.

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import mean_squared_error

warnings.filterwarnings("ignore", category=ConvergenceWarning)

iterations = range(1, 11)
train_sse, val_sse = [], []
train_mse, val_mse = [], []
train_rmse, val_rmse = [], []

# Запускаем тот же sparse_cg с разным ограничением по числу итераций
for n_iter in iterations:
    current_model = Ridge(
        alpha=0.0,
        solver="sparse_cg",
        tol=1e-12,
        max_iter=n_iter
    )
    current_model.fit(X_train, y_train)

    train_pred = current_model.predict(X_train)
    val_pred = current_model.predict(X_val)

    tr_mse = mean_squared_error(y_train, train_pred)
    vl_mse = mean_squared_error(y_val, val_pred)

    train_mse.append(tr_mse)
    val_mse.append(vl_mse)
    train_rmse.append(np.sqrt(tr_mse))
    val_rmse.append(np.sqrt(vl_mse))
    train_sse.append(np.sum((y_train.to_numpy() - train_pred) ** 2))
    val_sse.append(np.sum((y_val.to_numpy() - val_pred) ** 2))

plt.figure(figsize=(8, 4))
plt.plot(iterations, train_sse, marker="o", label="train")
plt.plot(iterations, val_sse, marker="o", label="val")
plt.xlabel("Итерация")
plt.ylabel("SSE")
plt.title("SSE по итерациям")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(iterations, train_mse, marker="o", label="train")
plt.plot(iterations, val_mse, marker="o", label="val")
plt.xlabel("Итерация")
plt.ylabel("MSE")
plt.title("MSE по итерациям")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(iterations, train_rmse, marker="o", label="train")
plt.plot(iterations, val_rmse, marker="o", label="val")
plt.xlabel("Итерация")
plt.ylabel("RMSE, минуты")
plt.title("RMSE по итерациям")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

print(f"Последний train RMSE: {train_rmse[-1]:.3f} мин")
print(f"Последний val RMSE:   {val_rmse[-1]:.3f} мин")

#### Вопрос 2.

Какую информацию про нашу модель несут эти графики?

**Ответ:**  
Они показывают, как меняется ошибка модели при обучении. В нашем случае обе ошибки быстро уменьшаются и потом почти не меняются, что является признаком нормального обучения.

#### Вопрос 3. 

Какая траектория линий train и val ошибок будет на графиках недообученной модели и почему?

Какая траектория линий train и val ошибок будет на графиках обученной модели и почему?

Какая траектория линий train и val ошибок будет на графиках переобученной модели и почему?

**Ответ:**  
Недообученность - train и val ошибки ещё большие и обычно продолжают снижаться — модель не успела найти хорошие веса.
Обученная - обе ошибки низкие и находятся близко друг к другу, дальше почти не меняются.
Переобученная - train ошибка продолжает уменьшаться, а val начинает расти. Значит модель всё лучше запоминает тренировочные данные, но хуже работает на новых.

#### Шаг 6. Применение модели на тестовой выборке

In [ ]:
test_predictions = model.predict(X_test)

result = pd.DataFrame({
    "Фактическое время": y_test.to_numpy(),
    "Прогноз": np.round(test_predictions, 2),
    "Ошибка": np.round(test_predictions - y_test.to_numpy(), 2),
})

display(result.head(10))

#### Шаг 7. RMSE на тестовой выборке

In [ ]:
test_mse = mean_squared_error(y_test, test_predictions)
test_rmse = np.sqrt(test_mse)

print(f"RMSE на тестовой выборке: {test_rmse:.3f} минуты")

#### Вопрос 4.

Что получившийся RMSE может сказать о нашей модели?

**Ответ:**  
RMSE показывает типичный размер ошибки прогноза в тех же единицах, что и ответ — здесь в минутах. Получилось примерно **4.8 минуты**. Это близко к validation RMSE, поэтому на тестовых данных модель работает примерно так же, как на валидационных, и сильного переобучения не видно.